# investalyze — sample records

15 random rows from each provider's table(s) in `investalyze.duckdb`.
Run from the repo root (so `ingest.toml` resolves). Providers not built yet show a note.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from investalyze.ingest import config, storage

display(HTML('<style>table.dataframe td {white-space: nowrap;}</style>'))
pd.set_option('display.max_columns', None)


# resolve the repo root (walk up to ingest.toml) so paths work regardless of CWD
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ingest.toml').exists())
cfg = config.load(root / 'ingest.toml')
con = storage.connect(root / cfg.data_root, read_only=True)  # explore only, never mutate
existing = {name for (name,) in con.execute('SHOW TABLES').fetchall()}

# provider -> its table(s); fill in as each provider is built
PROVIDER_TABLES = {
    'stooq': ['market_data'],
    'yahoo': ['prices', 'dividends', 'splits'],
    'simfin': ['income', 'balance', 'cashflow', 'companies'],
}


def sample(table, n=15):
    """15 random rows from `table` as a DataFrame."""
    return con.execute(f'SELECT * FROM {table} ORDER BY random() LIMIT {n}').df()


def show(provider):
    """Display a 15-row sample of every table the provider owns."""
    tables = PROVIDER_TABLES[provider]
    if not tables:
        print(f'{provider}: no tables yet (provider not built)')
        return
    for table in tables:
        if table not in existing:
            print(f'{provider}: `{table}` not in the DB yet')
            continue
        print(table)
        display(sample(table))

## Stooq — bonds, currencies, indices

In [2]:
show('stooq')

market_data


,Ticker,Date,O,H,L,C,AssetClass
0,XAGCNY,2016-11-30,115.019000,115.735000,114.538000,115.137000,currencies
1,HKDISK,2002-04-02,12.756000,12.756000,12.756000,12.756000,currencies
2,PLNRON,2018-12-04,1.086910,1.088630,1.084720,1.087160,currencies
3,CHFUSD,1996-11-12,0.792100,0.792100,0.792100,0.792100,currencies
4,TRYCNY,2018-06-08,1.426980,1.429900,1.412380,1.416700,currencies
5,CLPBRL,2008-03-25,0.003874,0.003874,0.003874,0.003874,currencies
6,^MT30,2018-04-18,1148.520000,1148.520000,1148.520000,1148.520000,indices
7,ARSBTC,2014-01-21,0.000168,0.000169,0.000167,0.000168,currencies
8,GBPMYR,2023-02-16,5.311270,5.317630,5.289640,5.308770,currencies
9,HUFCHF,2006-02-01,0.006179,0.006179,0.006179,0.006179,currencies


## Yahoo — stock prices

In [3]:
show('yahoo')

prices


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Ticker,Date,O,H,L,C,V,AC
0,NICE,2013-03-20,36.560001,36.860001,36.459999,36.630001,128500,34.803580
1,IRD,2024-02-08,2.510000,2.600000,2.450000,2.520000,288500,2.520000
2,MCN,2022-10-13,6.620000,7.050000,6.550000,6.930000,165500,4.649788
3,LUV,2015-09-16,38.759998,39.459999,38.500000,39.139999,6382400,34.446642
4,BGH,2014-05-21,24.410000,24.410000,24.190001,24.350000,61200,7.074530
5,TCPC,2020-10-22,9.650000,9.730000,9.600000,9.730000,107900,4.547610
6,CTM,2023-10-30,0.358000,0.500000,0.358000,0.500000,2520500,0.500000
7,WHWK,2022-08-04,13.620000,14.260000,13.620000,14.110000,117000,14.110000
8,RPM,1997-05-21,14.800000,14.800000,14.500000,14.750000,284875,6.082756
9,QETAU,2026-03-05,11.280000,11.280000,11.280000,11.280000,0,11.280000


dividends


,Ticker,Date,Dividend
0,SB,2013-02-28,0.050000
1,WFC-PY,2023-11-29,0.351560
2,TAP,1990-02-22,0.062500
3,NMT,2005-06-13,0.071000
4,WDTE,2024-03-01,1.800000
5,BBDO,2019-12-03,0.004000
6,NCV,2022-12-09,0.172000
7,AVNT,2025-03-18,0.270000
8,KMI,2011-04-28,0.140000
9,SEM,2021-11-15,0.067349


splits


,Ticker,Date,Ratio
0,BOKF,1996-11-14,1.030000
1,EHC,1992-01-02,1.500000
2,VZ,1986-04-18,2.000000
3,CDIO,2025-05-13,0.033333
4,ELPW,2026-03-12,0.012500
5,BYRN,2021-04-27,0.100000
6,HSDT,2018-01-23,0.200000
7,PKBK,2013-05-03,1.100000
8,TR,1995-07-12,2.000000
9,SNFCA,2022-06-30,1.050000


## SimFin — fundamentals

In [4]:
show('simfin')

income


,Ticker,SrcId,Src,Market,Period,IsRestated,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Revenue,Sales & Services Revenue,Financing Revenue,Other Revenue,Cost of Revenue,Cost of Goods & Services,Cost of Financing Revenue,Cost of Other Revenue,Gross Profit,Other Operating Income,Operating Expenses,"Selling, General & Administrative",Selling & Marketing,General & Administrative,Research & Development,Depreciation & Amortization,Provision for Doubtful Accounts,Other Operating Expenses,Operating Income (Loss),Non-Operating Income (Loss),"Interest Expense, Net",Interest Expense,Interest Income,Other Investment Income (Loss),Foreign Exchange Gain (Loss),Income (Loss) from Affiliates,Other Non-Operating Income (Loss),"Pretax Income (Loss), Adj.",Abnormal Gains (Losses),Acquired In-Process R&D,Merger & Acquisition Expense,Abnormal Derivatives,Disposal of Assets,Early Extinguishment of Debt,Asset Write-Down,Impairment of Goodwill & Intangibles,Sale of Business,Legal Settlement,Restructuring Charges,Sale of Investments & Unrealized Investments,Insurance Settlement,Other Abnormal Items,Pretax Income (Loss),"Income Tax (Expense) Benefit, Net",Current Income Tax,Deferred Income Tax,Tax Allowance/Credit,"Income (Loss) from Affiliates, Net of Taxes",Income (Loss) from Continuing Operations,Net Extraordinary Gains (Losses),Discontinued Operations,Accounting Charges & Other,Income (Loss) Incl. Minority Interest,Minority Interest,Net Income,Preferred Dividends,Other Adjustments,Net Income (Common)
0,FE,422040,simfin,us,Q,False,USD,2014,Q3,2014-09-30,2014-10-29,2015-10-29,420000000,421000000,3888000000,<NA>,<NA>,<NA>,-1732000000,<NA>,<NA>,<NA>,2156000000,<NA>,-1440000000,<NA>,<NA>,<NA>,<NA>,-343000000,<NA>,-1097000000,716000000,-231000000,-275000000,<NA>,<NA>,16000000,<NA>,<NA>,28000000,485000000,0,None,<NA>,<NA>,None,<NA>,None,<NA>,<NA>,None,<NA>,None,None,<NA>,485000000,-152000000,<NA>,<NA>,None,<NA>,333000000,0,0,<NA>,333000000,<NA>,333000000,<NA>,<NA>,333000000
1,POOL,448357,simfin,us,Q,False,USD,2005,Q1,2005-03-31,2005-04-29,2005-04-29,52273000,55494000,265161000,<NA>,<NA>,<NA>,-193210000,<NA>,<NA>,<NA>,71951000,<NA>,-60650000,-60650000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,11301000,-2562000,-1080000,<NA>,<NA>,<NA>,<NA>,-1482000,<NA>,8739000,<NA>,None,<NA>,<NA>,None,<NA>,None,<NA>,<NA>,None,<NA>,None,None,<NA>,8739000,-3986000,<NA>,<NA>,None,<NA>,4753000,<NA>,<NA>,<NA>,4753000,<NA>,4753000,<NA>,<NA>,4753000
2,NTUS,497305,simfin,us,Q,True,USD,2010,Q1,2010-03-31,2010-05-07,2011-05-06,27726000,27726000,49275000,<NA>,<NA>,<NA>,-19411000,<NA>,<NA>,<NA>,29864000,<NA>,-30104000,-24974000,<NA>,<NA>,-5130000,<NA>,<NA>,<NA>,-240000,-55000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-55000,-295000,<NA>,None,<NA>,<NA>,None,<NA>,None,<NA>,<NA>,None,<NA>,None,None,<NA>,-295000,-36000,<NA>,<NA>,None,<NA>,-331000,<NA>,<NA>,<NA>,-331000,<NA>,-331000,<NA>,<NA>,-331000
3,KVHI,9075880,simfin,us,Q,False,USD,2011,Q1,2011-03-31,2011-05-05,2011-05-05,14748000,14748000,24409000,<NA>,<NA>,<NA>,-15330000,<NA>,<NA>,<NA>,9079000,<NA>,-11101000,-8127000,<NA>,<NA>,-2974000,<NA>,<NA>,<NA>,-2022000,23000,9000,<NA>,<NA>,<NA>,<NA>,<NA>,14000,-1999000,<NA>,None,<NA>,<NA>,None,<NA>,None,<NA>,<NA>,None,<NA>,None,None,<NA>,-1999000,465000,<NA>,<NA>,None,<NA>,-1534000,<NA>,<NA>,<NA>,-1534000,<NA>,-1534000,<NA>,<NA>,-1534000
4,KAMN,7962698,simfin,us,A,True,USD,2006,FY,2006-12-31,2007-03-01,2009-02-26,24080303,24450769,991422000,<NA>,<NA>,<NA>,-719999000,<NA>,<NA>,<NA>,271423000,<NA>,-223549000,-223549000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,47874000,-7162000,-6244000,<NA>,<NA>,<NA>,<NA>,<NA>,-918000,40712000,-52000,None,<NA>,<NA>,None,<NA>,None,<NA>,<NA>,None,<NA>,None,None,<NA>,40660000,-16017000,<NA>,<NA>,None,<NA>,24643000,7143000,7143000,<NA>,31786000,<NA>,31786000,<NA>,<NA>,31786000
5,TA_delisted,133962,simfin,us,A,True,USD,2019,FY,2019-12-31,2020-02-25,2021-02-26,7783000,7783000,6117359000,<NA>,<NA>,<NA>,-4594769000,<NA>,<NA>,<NA>,1522590000,<N

balance


,Ticker,SrcId,Src,Market,Period,IsRestated,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),"Cash, Cash Equivalents & Short Term Investments",Cash & Cash Equivalents,Short Term Investments,Accounts & Notes Receivable,"Accounts Receivable, Net","Notes Receivable, Net",Unbilled Revenues,Inventories,Raw Materials,Work In Process,Finished Goods,Other Inventory,Other Short Term Assets,Prepaid Expenses,Derivative & Hedging Assets (Short Term),Assets Held-for-Sale,Deferred Tax Assets (Short Term),Income Taxes Receivable,Discontinued Operations (Short Term),Misc. Short Term Assets,Total Current Assets,"Property, Plant & Equipment, Net","Property, Plant & Equipment",Accumulated Depreciation,Long Term Investments & Receivables,Long Term Investments,Long Term Marketable Securities,Long Term Receivables,Other Long Term Assets,Intangible Assets,Goodwill,Other Intangible Assets,Prepaid Expense,Deferred Tax Assets (Long Term),Derivative & Hedging Assets (Long Term),Prepaid Pension Costs,Discontinued Operations (Long Term),Investments in Affiliates,Misc. Long Term Assets,Total Noncurrent Assets,Total Assets,Payables & Accruals,Accounts Payable,Accrued Taxes,Interest & Dividends Payable,Other Payables & Accruals,Short Term Debt,Short Term Borrowings,Short Term Capital Leases,Current Portion of Long Term Debt,Other Short Term Liabilities,Deferred Revenue (Short Term),Liabilities from Derivatives & Hedging (Short Term),Deferred Tax Liabilities (Short Term),Liabilities from Discontinued Operations (Short Term),Misc. Short Term Liabilities,Total Current Liabilities,Long Term Debt,Long Term Borrowings,Long Term Capital Leases,Other Long Term Liabilities,Accrued Liabilities,Pension Liabilities,Pensions,Other Post-Retirement Benefits,Deferred Compensation,Deferred Revenue (Long Term),Deferred Tax Liabilities (Long Term),Liabilities from Derivatives & Hedging (Long Term),Liabilities from Discontinued Operations (Long Term),Misc. Long Term Liabilities,Total Noncurrent Liabilities,Total Liabilities,Preferred Equity,Share Capital & Additional Paid-In Capital,Common Stock,Additional Paid in Capital,Other Share Capital,Treasury Stock,Retained Earnings,Other Equity,Equity Before Minority Interest,Minority Interest,Total Equity,Total Liabilities & Equity
0,TNL,116466,simfin,us,Q,True,USD,2007,Q3,2007-09-30,2007-11-08,2007-11-08,180000000,180000000,231000000,231000000,<NA>,419000000,419000000,<NA>,None,554000000,<NA>,<NA>,<NA>,<NA>,806000000,150000000,<NA>,<NA>,93000000,<NA>,<NA>,563000000,2010000000,978000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,7212000000,1035000000,2728000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3449000000,8190000000,10200000000,1075000000,342000000,<NA>,<NA>,733000000,159000000,<NA>,<NA>,<NA>,1023000000,580000000,<NA>,<NA>,<NA>,443000000,2257000000,3007000000,<NA>,<NA>,1528000000,<NA>,<NA>,<NA>,<NA>,<NA>,274000000,890000000,<NA>,<NA>,364000000,4535000000,6792000000,<NA>,3613000000,<NA>,<NA>,<NA>,-829000000,427000000,197000000,3408000000,<NA>,3408000000,10200000000
1,MCO,88874,simfin,us,Q,False,USD,2007,Q1,2007-03-31,2007-05-03,2007-05-03,277700000,284900000,299800000,289500000,10300000,499300000,499300000,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,80000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,80000000,879100000,118200000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,472600000,63500000,178300000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,230800000,590800000,1469900000,282900000,<NA>,<NA>,<NA>,282900000,80000000,<NA>,<NA>,<NA>,410100000,410100000,<NA>,<NA>,<NA>,<NA>,773000000,300000000,<NA>,<NA>,467000000,<NA>,<NA>,<NA>,<NA>,<NA>,107800000,<NA>,<NA>,<NA>,359200000,767000000,1540000000,<NA>,358100000,<NA>,<NA>,<NA>,-2643700000,2223400000,-7900000,-70100000,<NA>,-70100000,1469900000
2,WHD,6767475,simfin,us,Q,False,USD,2023,Q4,2023-12-31,2024-02-29,2024-02-29,65379000,108196000,133792000,133792000,<NA>,205381000,205381000,<NA>,None,205625000,<NA>,<NA>,<NA>,<NA>,11380000,11380000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,556178000,345502000,<NA>,<

cashflow


,Ticker,SrcId,Src,Market,Period,IsRestated,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Net Income/Starting Line,Net Income,Net Income from Discontinued Operations,Other Adjustments,Depreciation & Amortization,Non-Cash Items,Stock-Based Compensation,Deferred Income Taxes,Other Non-Cash Adjustments,Change in Working Capital,Change in Accounts Receivable,Change in Inventories,Change in Accounts Payable,Change in Other,Net Cash from Discontinued Operations (Operating),Net Cash from Operating Activities,Change in Fixed Assets & Intangibles,Disposition of Fixed Assets & Intangibles,Disposition of Fixed Assets,Disposition of Intangible Assets,Acquisition of Fixed Assets & Intangibles,Purchase of Fixed Assets,Acquisition of Intangible Assets,Other Change in Fixed Assets & Intangibles,Net Change in Long Term Investment,Decrease in Long Term Investment,Increase in Long Term Investment,Net Cash from Acquisitions & Divestitures,Net Cash from Divestitures,Cash for Acquisition of Subsidiaries,Cash for Joint Ventures,Net Cash from Other Acquisitions,Other Investing Activities,Net Cash from Discontinued Operations (Investing),Net Cash from Investing Activities,Dividends Paid,Cash from (Repayment of) Debt,"Cash from (Repayment of) Short Term Debt, Net","Cash from (Repayment of) Long Term Debt, Net",Repayments of Long Term Debt,Cash from Long Term Debt,Cash from (Repurchase of) Equity,Increase in Capital Stock,Decrease in Capital Stock,Other Financing Activities,Net Cash from Discontinued Operations (Financing),Net Cash from Financing Activities,Net Cash Before Disc. Operations and FX,Change in Cash from Disc. Operations and Other,Net Cash Before FX,Effect of Foreign Exchange Rates,Net Change in Cash
0,FET,124637,simfin,us,Q,True,USD,2013,Q4,2013-12-31,2015-03-01,2016-02-29,4589000,4736000,34545000,<NA>,<NA>,<NA>,16603000,19908000,<NA>,<NA>,<NA>,-23790000,<NA>,<NA>,<NA>,<NA>,<NA>,47266000,-15321000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-1000,<NA>,<NA>,<NA>,<NA>,0,<NA>,-15322000,<NA>,-18026000,<NA>,<NA>,<NA>,<NA>,-2776000,690000,-3466000,-1426000,<NA>,-22228000,9716000,<NA>,9716000,1665000,11381000
1,CALD,100006,simfin,us,Q,False,USD,2005,Q2,2005-06-30,2005-08-11,2005-08-11,26144000,26144000,-3718000,<NA>,<NA>,<NA>,414000,610000,<NA>,<NA>,<NA>,825000,<NA>,<NA>,<NA>,<NA>,<NA>,-1869000,-255000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2848000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2593000,<NA>,-149000,<NA>,<NA>,<NA>,<NA>,90000,90000,<NA>,<NA>,<NA>,-59000,665000,<NA>,665000,-121000,544000
2,AEM,2226695,simfin,us,A,False,USD,2012,FY,2012-12-31,2013-03-28,2013-03-28,171250000,171486000,310916000,<NA>,<NA>,<NA>,<NA>,405556000,<NA>,<NA>,<NA>,-20465000,<NA>,<NA>,<NA>,<NA>,<NA>,696007000,-445550000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,70645000,<NA>,<NA>,-9322000,<NA>,<NA>,<NA>,<NA>,8071000,<NA>,-376156000,-118121000,-102063000,<NA>,<NA>,<NA>,<NA>,20711000,<NA>,20711000,-3133000,<NA>,-202606000,117245000,<NA>,117245000,1376000,118621000
3,ACIW,1057787,simfin,us,Q,False,USD,2013,Q3,2013-09-30,2013-11-07,2013-11-07,117376000,119422000,13762000,<NA>,<NA>,<NA>,18677000,4062000,<NA>,<NA>,<NA>,-7572000,<NA>,<NA>,<NA>,<NA>,<NA>,28929000,-4732000,<NA>,<NA>,<NA>,-4732000,-2432000,-2300000,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-4732000,<NA>,101524000,<NA>,<NA>,<NA>,<NA>,-63677000,4903000,-68580000,-6298000,<NA>,31549000,55746000,<NA>,55746000,3024000,58770000
4,RLH,990923,simfin,us,A,False,USD,2018,FY,2018-12-31,2019-02-27,2020-02-27,24392000,25477000,14467000,<NA>,<NA>,<NA>,17003000,-25460000,<NA>,<NA>,<NA>,-9524000,<NA>,<NA>,<NA>,<NA>,<NA>,-3514000,105133000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-27249000,<NA>,<NA>,<NA>,<NA>,-986000,<NA>,76898000,<NA>,-67999000,<NA>,<NA>,<NA>,<NA>,236000,236000,<NA>,-30690000,<NA>,-98453000,-25069000,<NA>,-25069000,<NA>,-25069000
5,WRI,449242,simfin,us,Q,True,USD,2015,Q4,2015-12-31,2016-02-28,2018-02-28,123375000,121690000,49026000,<N

companies


,Ticker,SrcId,Src,Market,Industry,Sector,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,CIK,Main Currency
0,PEB,107219,simfin,us,REITs,Real Estate,Pebblebrook Hotel Trust,109001,US70509V1008,12,50,Pebblebrook Hotel Trust is an internally manag...,1474098,USD
1,LYFT,939173,simfin,us,Travel & Leisure,Consumer Cyclical,"Lyft, Inc.",103015,US55087P1049,12,4675,"Lyft, Inc. operates as an online social ridesh...",1759509,USD
2,TATT,18632555,simfin,us,Aerospace & Defense,Industrials,TAT Technologies Ltd.,100008,IL0010827264,12,471,"TAT Technologies Ltd., together with its subsi...",808439,USD
3,TPST,11035402,simfin,us,Biotechnology,Healthcare,"Tempest Therapeutics, Inc.",106002,US87978U1088,12,19,"Tempest Therapeutics Inc., a clinical-stage on...",1544227,USD
4,IPHA,18905018,simfin,us,Biotechnology,Healthcare,Innate Pharma S.A.,106002,US45781K2042,12,211,"Innate Pharma S.A., a biotechnology company, d...",1598599,USD
5,FGEN,781665,simfin,us,Biotechnology,Healthcare,FIBROGEN INC,106002,US31572Q8087,12,599,FibroGen Inc is a biopharmaceutical company. I...,921299,USD
6,AURA,1873135,simfin,us,Biotechnology,Healthcare,"Aura Biosciences, Inc.",106002,US05153U1079,12,70,"Aura Biosciences, Inc. operates as a biotechno...",1501796,USD
7,PRMW,17663714,simfin,us,Beverages - Non-Alcoholic,Consumer Defensive,Primo Water Corporation,102005,CA74167P1080,12,9240,Primo Water Corporation provides water direct ...,884713,USD
8,LOW,186050,simfin,us,Retail - Apparel & Specialty,Consumer Cyclical,LOWES COMPANIES INC,103002,US5486611073,1,2021,Lowe\'s Companies Inc is a home-improvement pr...,60667,USD
9,ASR,18720463,simfin,us,Transportation & Logistics,Industrials,"Grupo Aeroportuario del Sureste, S. A. B. de C...",100010,US40051E2028,12,1787,"Grupo Aeroportuario del Sureste, S. A. B. de C...",1123452,USD
